In [10]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [11]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [12]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [13]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [14]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [15]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [16]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [17]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 360,
 'tn': 2632,
 'fp': 5,
 'fn': 3,
 'misclassification_rate': 0.0026666666666666666,
 'false_positive_rate': 0.0018960940462646946,
 'false_negative_rate': 0.008264462809917356}

### Check results on the test set (new data not yet seen by the model)

In [18]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 35,
 'tn': 834,
 'fp': 40,
 'fn': 91,
 'misclassification_rate': 0.131,
 'false_positive_rate': 0.04576659038901602,
 'false_negative_rate': 0.7222222222222222}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

Based on the misclaasification rate, I would be 87% confident that I would be able to predict a bot given a random account. However, we can see that the false negative rate is very high, meaning that if an account is actually a bot, my model would actually be very bad at detecting them.  

### What are potential ramifications of false positives from the model?

The ramifications of a false positive is that an account that isn't actually a bot would be considered as one. This means that real account could end up being banned off of a social media site. This in my opinion, would be the worst case scenario, as it's very frustrating as a user to be falsely banned for being a bot. If a real user ends up getting banned mistakenly, this could actively prevent the user from using the social media site again due to mistrust in the bot detection algorithm.   

### What are potential ramifications of false negatives from the model?

The ramifications of a false negative is that an account that is actually a bot would not be considered as one. This means that bot accounts would not be banned off a social media site. This could mean that harmful narratives are pushed on social media due to the opinions of the few running the bot accounts. However as a social media site this is not as bad of an outcome as before, as bot accounts still bring engagement to the platform. The trade-off for this is that the overall quality of posts might tend to decrease over time, as bots usually tend to reguritate the same information over and over.   